### Tourism Data Preparation (Monthly Aggregation)

This notebook consolidates tourism data across years and standardizes it into a unified monthly dataset.

### Objectives:
- Merge yearly tourism data into a single dataset
- Ensure schema consistency across files
- Clean numeric inconsistencies (e.g., comma-formatted values)
- Aggregate to prefecture-level monthly demand

### Target Variable:
- `overnight_stays`: Monthly tourism demand proxy

### Output:
data/tourism_monthly/tourism_monthly_2016_2024.csv

In [ ]:
import pandas as pd 
import numpy as np
import os

In [ ]:
TOURISM_DIR = "data/tourism_monthly_by_year_new"   
OUTPUT_DIR = "data/tourism_monthly"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
final=[]
EXPECTED_COLS = {"prefecture", "year", "month", "overnight_stays"}

for fname in sorted(os.listdir(TOURISM_DIR)):
    if not fname.endswith(".csv"):
        continue
    
    path = os.path.join(TOURISM_DIR,fname)
    df = pd.read_csv(path)
    df.columns = (
    df.columns
      .str.strip()        # remove leading/trailing spaces
      .str.lower()        # make lowercase
    )
    # Schema validation
    missing = EXPECTED_COLS  - set(df.columns)
    if missing:
        raise ValueError(f"{fname} missing columns: {missing}")

    final.append(df)
    
    print(f"Loaded {fname}: {df.shape}")
    

In [ ]:
# Merge datasets
tourism_all = pd.concat(final, ignore_index=True)

#tourism_all.head(), tourism_all.shape

In [ ]:
# Normalize numeric format 
tourism_all["overnight_stays"] = (
    tourism_all["overnight_stays"]
    .astype(str)
    .str.replace(",", "")
    .astype(float)
)

In [ ]:
tourism_all.duplicated(
    subset=["prefecture", "year", "month"]
).sum()

In [ ]:
# Aggregate demand at prefecture-month level
tourism_monthly_final = (
    tourism_all
    .groupby(["prefecture", "year", "month"], as_index=False)
    .agg(
        overnight_stays=("overnight_stays", "sum")
    )
)


In [ ]:
tourism_monthly_final.duplicated(
    subset=["prefecture", "year", "month"]
).sum()


In [ ]:
final_tourism_path = os.path.join(
    OUTPUT_DIR, "tourism_monthly_2016_2024.csv"
)

tourism_monthly_final.to_csv(final_tourism_path, index=False)
